In [16]:
# Extract_Cemetery_Plot_Data.py
# 9/11/2025

# ** Input ND Cemetery graves data with plot info
# *****Extract Jewish sections
# *****Extract Jewish Graves from ND Cemeteries



import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
import json
import sys
import random
import numpy as np

from dateutil import parser
from dateutil.relativedelta import relativedelta
import ast


def maximize_screen():
    from IPython.display import display,HTML
    display(HTML("<style>:root{--jp-notebook-max-width:100%!important;}</style>"))

def  timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':|',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def  display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        input('')

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC','TEC5'\
                'PVT','S Sgt','Capt','Pfc','TSgt','COL','Tec5','TEC4','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant','TEC 5','Maj','GM3','TEC 4','Tec4','RM3','Corp','T 5','MS','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df      

def discard_empty_sets(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:x.discard('') or x if isinstance(x,set) else x)
    return dataframe_df

def  trim_dataframe_set(dataframe_df,column):
    dataframe_df[column]=dataframe_df[column].apply(
            lambda x:{item.strip() for item in x if isinstance(item,str)} if isinstance(x,set) else x)
    return dataframe_df

def match_sets(soldier_match_list_df,search,column_search): 
    search_set= ast.literal_eval(search)
    if isinstance(soldier_match_list_df[column_search].iloc[0], set)==False:
        soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('{') else set()) 
    soldier_match_list_df.loc[:,column_search] = soldier_match_list_df[column_search].apply(
        lambda x: {'nan'} if pd.isna(x) else x)
    soldier_match_list_df = soldier_match_list_df.copy()
    soldier_match_list_df.loc[:,'set_difference']=search_set-soldier_match_list_df[column_search]
    return soldier_match_list_df

def discard_set_one_character(data_set_df,column):
    data_set_df=data_set_df.copy()
    data_set_df[column] = data_set_df[column].apply(
        lambda x: {item for item in ast.literal_eval(x) if len(item) > 2} if isinstance(x, str) else 
              ({item for item in x if len(item) > 1} if isinstance(x, set) else x))
    return data_set_df

def drop_sets_dataframe_duplicates(dataframe_set_df):
    set_columns = [col for col in dataframe_set_df.columns]
    # print(set_columns)
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: tuple(sorted(x)) if isinstance(x, set) else x)   
    dataframe_set_df = dataframe_set_df.drop_duplicates()
    dataframe_set_df=dataframe_set_df.copy()
    for col in set_columns:
        dataframe_set_df[col] = dataframe_set_df[col].apply(
            lambda x: set(x) if isinstance(x, tuple) else x )
    return dataframe_set_df

def convert_dataframe_to_set(dataframe_df,column,delimiter):
    column_set=column+'_set'
    dataframe_df_split=dataframe_df[column].str.split(delimiter)
    dataframe_df[column_set]=dataframe_df_split.apply(set)
    dataframe_df=discard_empty_sets(dataframe_df,column_set)
    dataframe_df=trim_dataframe_set(dataframe_df,column_set)   
    dataframe_df=discard_set_one_character(dataframe_df,column_set)
    return dataframe_df

def clean_name_set(name_set):
    if not isinstance(name_set, set):
        return set()
    return {
        name.strip()
        for name in name_set
        if isinstance(name, str) and len(name.strip()) > 1}

def clean_name_set_column(name_set):
    if not isinstance(name_set, set):
        return set()
    cleaned_names = set()
    for name in name_set:
        if not isinstance(name, str):
            continue
        # Split name into parts, remove single letters and empty strings
        parts = [p.strip() for p in name.split() if len(p.strip()) > 1]
        if parts:
            cleaned_names.add(' '.join(parts))  # Recombine cleaned parts
    return cleaned_names

def split_set_into_sets(dataframe_set_df,column):
    dataframe_set_df=dataframe_set_df.copy()
    dataframe_set_df[column] = dataframe_set_df[column].apply(
    lambda x: {part.strip() for item in x if isinstance(item, str) for part in item.split() if part.strip()} if isinstance(x, set) else set())
    return dataframe_set_df

def dataframe_contains_match(dataframe_df,search_column,match_column):
    column_out=search_column+'_match'
    dataframe_df[column_out]=dataframe_df.apply(
        lambda row: row[match_column] in row[search_column] if isinstance(row[search_column], str) else False,axis=1)
    return dataframe_df



def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r'[^a-z\s]', '', s)  # keep only letters and spaces
    s = re.sub(r'\s+', ' ', s)      # collapse multiple spaces
    return s.strip()


def haystack_contains_needle(needle_df,haystack_df):
    mask1 = [
    normalize(parent) in normalize(spouse)
    for parent,spouse in zip(
        needle_df,
        haystack_df,  
    )
    ]
    return mask1



def  match_2_set_columns(searcher_df,searchee_df,searcher_column,searchee_column):
    # searcher_df_split=searcher_df[searcher_column].str.split(' ')
    # searcher_df=searcher_df_split.apply(set)   
    explode_searchee_df=searchee_df.explode(searchee_column)
    print('explode_searchee_df:',explode_searchee_df)
    explode_searchee_df_split=explode_searchee_df[searchee_column].str.split(' ')
    print('\nexplode_searchee_df_split:',explode_searchee_df_split)
    explode_searchee_df=explode_searchee_df_split.apply(set)
    print('\nexplode_searchee_df:',explode_searchee_df)
    print('\nsearcher_df[searcher_column]:',searcher_df[searcher_column])
    
    searcher_matches=searcher_df[searcher_column]-explode_searchee_df
    searcher_matches
    return searcher_matches



def match_soldier_to_child(soldier_df,child_df,soldier_set_column,children_set_column):
    match_set=match_2_set_columns(soldier_df,child_df,soldier_set_column,children_set_column)  
    child_matches_df=pd.DataFrame(match_set,columns=['child_match'])
    match_index = child_matches_df[child_matches_df['child_match']==set()].index
    
    # Initialized Column to Boolian
    soldier_df['children_match']='False'
    soldier_df.loc[match_index, 'children_match'] ='True'
    return soldier_df
    
def single_parent_child_match(parent):
    global soldiers_df,abmc_fag_df
    soldiers_df['Parent_1']=soldiers_df['Parents'].str.split('*').str[0].str.strip()
    soldiers_df['Parent_2']=soldiers_df['Parents'].str.split('*').str[1].str.strip()
    soldiers_df1=soldiers_df[['name_index','soldiers_full_name2','Parent_1','Parent_2','birth_year','soldiers_cemetery','entry_state']]
    soldiers_df1.columns=['name_index','soldiers_full_name2','Parent_1','Parent_2','soldiers_birth_year','soldiers_cemetery','entry_state']

    
    abmc_fag_df1=abmc_fag_df[['full_name','spouses','birth_year','children','cemetery_name','cemetery_id','prob_jewish'\
                              ,'low_grave_count','memorial_count','birth_state','birth_country','plot']]
    abmc_fag_df1.columns=['full_name','spouses','parent_birth_year','children','cemetery_name','cemetery_id','prob_jewish'\
                              ,'low_grave_count','memorial_count','birth_state','birth_country','plot']
    
    matched_parent_spouse_df11=pd.merge(
        soldiers_df1,
        abmc_fag_df1,
        left_on=[parent],
        right_on=['full_name'],    
        how='inner') 
    matched_parent_spouse_df11['Parent']=matched_parent_spouse_df11[parent]

    print('matched_parent_spouse_df11-Size:',len(matched_parent_spouse_df11))
    matched_parent_spouse_df11=drop_sets_dataframe_duplicates(matched_parent_spouse_df11)
    matched_parent_spouse_df11.reset_index(drop=True, inplace = True)
    
    matched_parent_spouse_df11=clean_dataframe(matched_parent_spouse_df11,'soldiers_full_name2')
    matched_parent_spouse_df11=clean_dataframe(matched_parent_spouse_df11,'children')
    
    needle='soldiers_full_name2'
    haystack='children'
    dataframe_df=matched_parent_spouse_df11
    matched_parent_spouse_df11, mask=haystack_contains_needle_fuzzy(dataframe_df,needle,haystack)
    
    matched_out_df11c=pd.merge(
        soldiers_df1,
        matched_parent_spouse_df11,
        left_on=[ 'name_index'],
        right_on=['name_index'],
        how='inner')
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    matched_out_df11c=drop_sets_dataframe_duplicates(matched_out_df11c)
    matched_out_df11c.reset_index(drop=True, inplace = True)
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    
    matched_out_df11c=drop_sets_dataframe_duplicates(matched_out_df11c)
    matched_out_df11c.reset_index(drop=True, inplace = True)
    print('matched_out_df11c-Size:',len(matched_out_df11c))
    
    reorder_format=['name_index','soldiers_full_name2_x','Parent','spouses','full_name','children_match','prob_jewish','low_grave_count', 'soldiers_birth_year_x', 'parent_birth_year','birth_state'\
      ,'entry_state_x','birth_country','children','cemetery_name','cemetery_id','memorial_count','soldiers_cemetery_x','plot']
    
    rename_format=['name_index','soldiers_full_name','parent','spouse','full_name','children_match','prob_jewish','low_grave_count', 'soldiers_birth_year', 'parent_birth_year','birth_state'\
                    ,'entry_state','birth_country','children','cemetery_name','cemetery_id','memorial_count','soldiers_cemetery','plot']
    matched_out_df11c=matched_out_df11c[reorder_format]
    matched_out_df11c.columns=rename_format
    matched_out_df11c['Age']=matched_out_df11c['soldiers_birth_year']-matched_out_df11c['parent_birth_year'].astype('Int64')
    return matched_out_df11c
    
# **** Single_Parent_Child_Match Parent 1 *******


def  search_dataframe(dataframe,column,search_text):
    dataframe_extract=dataframe[dataframe[column]==search_text]
    return dataframe_extract

def clean_dataframe(dataframe_df,column):
    delete_list=['Pvt ','PTE ','Dr. ','Rev ','MAJ ','GEN ','Mrs ','CPT ',',PFC ','SGT ','Col ','LT ','Dr ',\
                ' Sr',' Jr','\.','Minor ','Infant ','Boy ','Girl ','Twins ','PFC ','TEC5 ',\
                'PVT ','S Sgt ','Capt ','Pfc ','TSgt ','COL ','Tec5 ','TEC4 ','1','2','III','II','Lt','CPL','SSgt','Sgt',\
                'Staff Sergeant ','TEC 5 ','Maj ','GM3 ','TEC 4 ','Tec4 ','RM3 ','Corp ','T 5 ','MS ','  ']
    dataframe_df=dataframe_df.copy()
    for item in delete_list:
        dataframe_df[column]= dataframe_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df 

def normalize_plot(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[*:,-]', ' ', text)        # replace separators with space
    text = re.sub(r'\s+', ' ', text)           # collapse multiple spaces
    return text.strip()
    
def normalize_plot_letter(text):
    if pd.isna(text):
        return 
    text = text.upper()
    text = re.sub(r'[-]', '', text)
    # replace separators with space
    return text.strip()
    

def extract_section(text):
    # match variations like "sec q", "sec-q", "section l"  
    pattern = r'(sec(?:tion)?)[\s\-\.:]*([a-z0-9]+)'
    match = re.search(pattern, text, re.I)

    # match = re.search(r'(sec(?:tion)?)[\s\-:]*([a-z0-9]+)', text, re.I)
    if match:
        return f"Section {match.group(2).upper()}"
    return None



def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    # dataframe_df[needle]=normalize(dataframe_df[needle])
    match=haystack+'_match' 
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def all_parts_in(needle, haystack):
    if pd.isna(haystack):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def extract_section_code(text):
    if pd.isna(text):
        return None
    # Remove * and extra spaces
    text = re.sub(r'[*]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Case 1: Starts like "A-19 ..." -> keep only "A"
    m = re.match(r'^([A-Za-z]+[\-\s]?\d+)', text)
    section_letter = m.group(1).upper() if m else None
    section_letter = re.sub(r'[\-\s].+', '', m.group(1)).strip() if m else None
    # if m:
    #     print('Case 1:',text,section_letter)
    #     input('')
    if m:
        return  f"Section {section_letter}"

    # Case 3: Starts like "M1-31B-2" -> keep "M1"
    m = re.match(r'^([A-Za-z]\s)', text)
    # if m:
    #     print('Case 3:',text,m.group(1).upper())
    #     input('')
    if m and len(m.group(1).upper()) <=2 and len(m.group(1).upper())>0:
        return  f"Section {m.group(1).upper()}"
        
    # Case 2: Starts like "M1-31B-2" -> keep "M1"
    m = re.match(r'^([A-Za-z]\d+)', text)
    # if m:
    #     print('Case 2:',text,m.group(1).upper())
    #     input('')
    if m:
        return  f"Section {m.group(1).upper()}"
    return None

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Extract_Cemetery_Plot_Data'
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')

DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"



# fileInput="ND_Cemeteries_All_Graves_Run_2_01-03"
fileInput="Non_Denominational_Cemeteries_All_Graves_All"

fileInputPhrases="ND_Sections_Worksheet"
fileInputJewishNames="Jewish_Name_List_Expanded"
fileInputJewishKeywords="Jewish_Keywords"
fileOutput="ND_Cemeteries_Plot_Extraction_All"

fileIn=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileInput+".csv"
fileInPhrases=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputPhrases+".csv"
fileInJewishNames=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputJewishNames+".csv"
fileInJewishKeywords=DirMain+"Affilation_Analysis\\Inputs\\"+fileInputJewishKeywords+".csv"

fileOut=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+"_"+timestamp+".csv"
fileOut0=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\"+fileOutput+".csv"

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Read In %%%%%%%%%%%%%%
graves_df=pd.read_csv(fileIn,encoding='ISO-8859-1', on_bad_lines='skip',low_memory=False)
print('• graves_df * Download Complete *, Size:',len(graves_df))
phrases_df=pd.read_csv(fileInPhrases,encoding='ISO-8859-1', on_bad_lines='skip')
print('• phrases_df * Download Complete *, Size:',len(phrases_df))
jewish_names_df=pd.read_csv(fileInJewishNames,encoding='ISO-8859-1', on_bad_lines='skip')
print('• Jewish_Names_df * Download Complete *, Size:',len(jewish_names_df))
jewish_keywords_df=pd.read_csv(fileInJewishKeywords,encoding='ISO-8859-1', on_bad_lines='skip')
print('• jewish_keywords_df * Download Complete *, Size:',len(jewish_keywords_df))
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% Read In %%%%%%%%%%%%%%

phrases_df_list=['cemetery_id','cemetery_name','jewish_section']
graves_df_list_all=['lastName','firstName','middleName','fullName','nickName','hasPlot','plot','birthDate','birthMonth','birthYear','deathDate','deathMonth','deathYear','birthCityName'\
  ,'birthStateName','birthCountryName','cemeteryName','countryName','cemeteryId','links','intermentHasPhoto','Parents','Spouses','Siblings','Children','prob_jewish']
graves_list_df=['lastName','cemeteryName','plot','cemeteryId']

graves_df_01=graves_df
graves_df_01 = graves_df_01.dropna(subset=['plot'])
graves_df_01.reset_index(drop=True, inplace = True)

graves_df_01 = graves_df_01.copy()
graves_df_01['cemeteryId_int'] = pd.to_numeric(graves_df_01['cemeteryId'].fillna(10000000),errors='coerce').astype('Int64')

graves_df_01 = graves_df_01.reset_index().rename(columns={"index": "graves_index"})
jewish_keywords_merge_df = graves_df_01.merge(
    jewish_keywords_df,
    left_on=["cemeteryId_int"],
    right_on=["cemeteryId"],
    how="inner"
)


jewish_keywords_plot_df, mask =haystack_contains_needle_fuzzy(jewish_keywords_merge_df,'keyword','plot')
drop_index=jewish_keywords_merge_df[jewish_keywords_merge_df['plot'].isna()].index
jewish_keywords_merge_df = jewish_keywords_merge_df.drop(index=drop_index)
mask_clean = mask.drop(index=drop_index)
jewish_keywords_plot_inner_df=jewish_keywords_merge_df[mask_clean]
graves_df_01.loc[jewish_keywords_plot_inner_df["graves_index"], "tags"] = jewish_keywords_plot_inner_df["tags"].values

# fileSectionsOut=DirMain+"Affilation_Analysis\\Outputs\\jewish_keywords_plot_inner_df.csv"
# jewish_keywords_plot_inner_df.to_csv(fileSectionsOut,mode="w",index=False,header=True)



graves_plot_extracted_df = graves_df_01.copy()
graves_plot_extracted_df["normalized"] = graves_plot_extracted_df["plot"].apply(normalize_plot_letter)


# print(graves_plot_extracted_df["plot"].apply(extract_section_code))
graves_plot_extracted_df["letter_section"]=graves_plot_extracted_df["plot"].apply(extract_section_code)
graves_plot_extracted_df["section"] = graves_plot_extracted_df["normalized"].apply(extract_section)

# Merge all tags in prioritized order
for col in ["letter_section", "section", "tags"]:
    mask = graves_plot_extracted_df[col].notna()
    graves_plot_extracted_df.loc[mask, "Merged"] = graves_plot_extracted_df.loc[mask, col]

print(graves_plot_extracted_df["Merged"])

fileSectionsOut=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\graves_plot_extracted_df_All.csv"
graves_plot_extracted_df.to_csv(fileSectionsOut,mode="w",index=False,header=True)
print('$$$$$$ Process Completed $$$$$$$$$, graves_plot_extracted_df uploaded to:',fileSectionsOut)

# Match to Jewish ND Sections and extract 
fileSectionsIn=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Inputs\\ND_Jewish_Sections.csv"
ND_Jewish_Sections_df=pd.read_csv(fileSectionsIn,encoding='ISO-8859-1', on_bad_lines='skip')
print('• jewish_keywords_df * Download Complete *, Size:',len(ND_Jewish_Sections_df))
graves_plot_extracted_df = graves_plot_extracted_df.drop_duplicates()
graves_plot_extracted_df.reset_index(drop=True, inplace = True)

ND_jewish_graves_df = graves_plot_extracted_df.merge(
    ND_Jewish_Sections_df,
    left_on=["cemeteryId_int"],
    right_on=["cemetery_id"],
    how="inner"
)

ND_jewish_graves_matched_df_list=['lastName', 'firstName', 'middleName', 'fullName', 'nickName', 'birthDate', 'birthMonth', 'birthYear'\
 , 'deathDate', 'deathMonth', 'deathYear', 'birthCityName', 'birthStateName', 'birthCountryName', 'cemeteryName', 'countryName'\
 , 'cemeteryId', 'links', 'intermentHasPhoto', 'Parents', 'Spouses', 'Siblings', 'Children', 'prob_jewish','Merged']    

ND_jewish_graves_matched_df, mask =haystack_contains_needle_fuzzy(ND_jewish_graves_df,'jewish_tag','Merged')
ND_jewish_graves_matched_df=ND_jewish_graves_matched_df[ND_jewish_graves_matched_df['Merged_match']==True]
ND_jewish_graves_matched_df = ND_jewish_graves_matched_df.drop_duplicates()
ND_jewish_graves_matched_df.reset_index(drop=True, inplace = True)
ND_jewish_graves_matched_df=ND_jewish_graves_matched_df[ND_jewish_graves_matched_df_list]
ND_jewish_graves_matched_df.rename(columns={'Merged': 'jewish_section'}, inplace=True)

ND_jewish_graves_matched_file=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Inputs\\ND_jewish_graves_matched_df.csv"
ND_jewish_graves_matched_df.to_csv(ND_jewish_graves_matched_file,mode="w",index=False,header=True)
print('$$$$$$ Process Completed $$$$$$$$$, ND_jewish_graves_matched_df uploaded to:',ND_jewish_graves_matched_file)







**Starting ***  Extract_Cemetery_Plot_Data  *** , Time: 2025-09-17 13:27:25.362502  **

• graves_df * Download Complete *, Size: 1094813
• phrases_df * Download Complete *, Size: 167
• Jewish_Names_df * Download Complete *, Size: 157
• jewish_keywords_df * Download Complete *, Size: 422
0            Section 11
1            Section 10
2             Section 3
3             Section 3
4            Section 11
              ...      
566998              NaN
566999        Section T
567000       Section 14
567001       Section 14
567002    Eternal Light
Name: Merged, Length: 567003, dtype: object
$$$$$$ Process Completed $$$$$$$$$


In [31]:
# Match to Jewish ND Sections and extract 
fileSectionsIn=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Inputs\\ND_Jewish_Sections.csv"
ND_Jewish_Sections_df=pd.read_csv(fileSectionsIn,encoding='ISO-8859-1', on_bad_lines='skip')
print('• jewish_keywords_df * Download Complete *, Size:',len(ND_Jewish_Sections_df))
graves_plot_extracted_df = graves_plot_extracted_df.drop_duplicates()
graves_plot_extracted_df.reset_index(drop=True, inplace = True)

ND_jewish_graves_df = graves_plot_extracted_df.merge(
    ND_Jewish_Sections_df,
    left_on=["cemeteryId_int"],
    right_on=["cemetery_id"],
    how="inner"
)

ND_jewish_graves_matched_df_list=['lastName', 'firstName', 'middleName', 'fullName', 'nickName', 'birthDate', 'birthMonth', 'birthYear'\
 , 'deathDate', 'deathMonth', 'deathYear', 'birthCityName', 'birthStateName', 'birthCountryName', 'cemeteryName', 'countryName'\
 , 'cemeteryId', 'links', 'intermentHasPhoto', 'Parents', 'Spouses', 'Siblings', 'Children', 'prob_jewish','Merged']    

ND_jewish_graves_matched_df, mask =haystack_contains_needle_fuzzy(ND_jewish_graves_df,'jewish_tag','Merged')
ND_jewish_graves_matched_df=ND_jewish_graves_matched_df[ND_jewish_graves_matched_df['Merged_match']==True]
ND_jewish_graves_matched_df = ND_jewish_graves_matched_df.drop_duplicates()
ND_jewish_graves_matched_df.reset_index(drop=True, inplace = True)
ND_jewish_graves_matched_df=ND_jewish_graves_matched_df[ND_jewish_graves_matched_df_list]
ND_jewish_graves_matched_df.rename(columns={'Merged': 'jewish_section'}, inplace=True)

ND_jewish_graves_matched_file=DirMain+"0-Master Worksheets\\Parent-Soldier Processing\\Inputs\\ND_jewish_graves_matched_df.csv"
ND_jewish_graves_matched_df.to_csv(ND_jewish_graves_matched_file,mode="w",index=False,header=True)
print('$$$$$$ Process Completed $$$$$$$$$')

• jewish_keywords_df * Download Complete *, Size: 179
$$$$$$ Process Completed $$$$$$$$$


In [24]:
mask

0           True
1           True
2          False
3          False
4           True
           ...  
1226772    False
1226773    False
1226774     True
1226775     None
1226776    False
Name: Merged_match, Length: 1226777, dtype: object

In [15]:
def normalize_plot(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[*:,-]', ' ', text)        # replace separators with space
    text = re.sub(r'\s+', ' ', text)           # collapse multiple spaces
    return text.strip()
    
def normalize_plot_letter(text):
    if pd.isna(text):
        return 
    text = text.upper()
    text = re.sub(r'[-]', '', text)
    # replace separators with space
    return text.strip()
    

def extract_section(text):
    # match variations like "sec q", "sec-q", "section l"  
    pattern = r'(sec(?:tion)?)[\s\-\.:]*([a-z0-9]+)'
    match = re.search(pattern, text, re.I)

    # match = re.search(r'(sec(?:tion)?)[\s\-:]*([a-z0-9]+)', text, re.I)
    if match:
        return f"Section {match.group(2).upper()}"
    return None

def extract_map(text):
    # match variations like "sec q", "sec-q", "section l"  
    pattern = r'(sec(?:tion)?)[\s\-\.:]*([a-z0-9]+)'
    match = re.search(pattern, text, re.I)

    # match = re.search(r'(sec(?:tion)?)[\s\-:]*([a-z0-9]+)', text, re.I)
    if match:
        return f"Map {match.group(2).upper()}"
    return None


def  haystack_contains_needle_fuzzy(dataframe_df,needle,haystack):
    # dataframe_df[needle]=normalize(dataframe_df[needle])
    match=haystack+'_match' 
    dataframe_df[match] = dataframe_df.apply(lambda row: all_parts_in(row[needle], row[haystack]),axis=1)
    mask=dataframe_df[match]
    return dataframe_df, mask

def all_parts_in(needle, haystack):
    if pd.isna(haystack):
        return None
    needle_parts = needle.split()   # split needle into words
    match_parts = all(part in haystack for part in needle_parts)
    return match_parts

def extract_section_code(text):
    if pd.isna(text):
        return None
    # Remove * and extra spaces
    text = re.sub(r'[*]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Case 1: Starts like "A-19 ..." -> keep only "A"
    m = re.match(r'^([A-Za-z]+[\-\s]?\d+)', text)
    section_letter = m.group(1).upper() if m else None
    section_letter = re.sub(r'[\-\s].+', '', m.group(1)).strip() if m else None
    # if m:
    #     print('Case 1:',text,section_letter)
    #     input('')
    if m:
        return  f"Section {section_letter}"

    # Case 3: Starts like "M1-31B-2" -> keep "M1"
    m = re.match(r'^([A-Za-z]\s)', text)
    # if m:
    #     print('Case 3:',text,m.group(1).upper())
    #     input('')
    if m and len(m.group(1).upper()) <=2 and len(m.group(1).upper())>0:
        return  f"Section {m.group(1).upper()}"
        
    # Case 2: Starts like "M1-31B-2" -> keep "M1"
    m = re.match(r'^([A-Za-z]\d+)', text)
    # if m:
    #     print('Case 2:',text,m.group(1).upper())
    #     input('')
    if m:
        return  f"Section {m.group(1).upper()}"
    return None



graves_df_01=graves_df
graves_df_01 = graves_df_01.dropna(subset=['plot'])
graves_df_01.reset_index(drop=True, inplace = True)

graves_df_01 = graves_df_01.copy()
graves_df_01['cemeteryId_int'] = pd.to_numeric(graves_df_01['cemeteryId'].fillna(10000000),errors='coerce').astype('Int64')

graves_df_01 = graves_df_01.reset_index().rename(columns={"index": "graves_index"})
jewish_keywords_merge_df = graves_df_01.merge(
    jewish_keywords_df,
    left_on=["cemeteryId_int"],
    right_on=["cemeteryId"],
    how="inner"
)


jewish_keywords_plot_df, mask =haystack_contains_needle_fuzzy(jewish_keywords_merge_df,'keyword','plot')
drop_index=jewish_keywords_merge_df[jewish_keywords_merge_df['plot'].isna()].index
jewish_keywords_merge_df = jewish_keywords_merge_df.drop(index=drop_index)
mask_clean = mask.drop(index=drop_index)
jewish_keywords_plot_inner_df=jewish_keywords_merge_df[mask_clean]
graves_df_01.loc[jewish_keywords_plot_inner_df["graves_index"], "tags"] = jewish_keywords_plot_inner_df["tags"].values

# fileSectionsOut=DirMain+"Affilation_Analysis\\Outputs\\jewish_keywords_plot_inner_df.csv"
# jewish_keywords_plot_inner_df.to_csv(fileSectionsOut,mode="w",index=False,header=True)



graves_plot_extracted_df = graves_df_01.copy()
graves_plot_extracted_df["normalized"] = graves_plot_extracted_df["plot"].apply(normalize_plot_letter)


# print(graves_plot_extracted_df["plot"].apply(extract_section_code))
graves_plot_extracted_df["letter_section"]=graves_plot_extracted_df["plot"].apply(extract_section_code)
graves_plot_extracted_df["section"] = graves_plot_extracted_df["normalized"].apply(extract_section)
graves_plot_extracted_df["map"] = graves_plot_extracted_df["normalized"].apply(extract_map)

# Merge all tags in prioritized order
for col in ["letter_section", "section", "tags"]:
    mask = graves_plot_extracted_df[col].notna()
    graves_plot_extracted_df.loc[mask, "Merged"] = graves_plot_extracted_df.loc[mask, col]

print(graves_plot_extracted_df["Merged"])

fileSectionsOut=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\graves_plot_extracted_df_All.csv"
graves_plot_extracted_df.to_csv(fileSectionsOut,mode="w",index=False,header=True)
print('$$$$$$ Process Completed $$$$$$$$$')
input('')


graves_df_01_jewish_names = graves_df_01.merge(
    jewish_names_df,
    left_on=["lastName"],
    right_on=["lastName"],
    how="inner"
)

fileSectionsOut=DirMain+"0-Master Worksheets\\Cemeteries\\Outputs\\graves_df_jewish_names_All.csv"
graves_df_01_jewish_names.to_csv(fileSectionsOut,mode="w",index=False,header=True)
print('$$$$$$ Process Completed $$$$$$$$$')

0            Section 11
1            Section 10
2             Section 3
3             Section 3
4            Section 11
              ...      
566998              NaN
566999        Section T
567000       Section 14
567001       Section 14
567002    Eternal Light
Name: Merged, Length: 567003, dtype: object
$$$$$$ Process Completed $$$$$$$$$


KeyboardInterrupt: Interrupted by user

In [ ]:
# graves_plot_extracted_df["normalized"]
graves_df = graves_df.drop_duplicates()
len(graves_df)

In [ ]:
for col in ["letter_section", "section", "map"]:   
    print(graves_plot_extracted_df[col].iloc[0:5])
    mask = graves_plot_extracted_df[col].notna()
    print(mask.iloc[0:5])
    input('')
    graves_plot_extracted_df.loc[mask, "Merged"] = graves_plot_extracted_df.loc[mask, col]
    print(graves_plot_extracted_df.loc[mask, "Merged"].iloc[0:50])
    input('')


# print(graves_plot_extracted_df["letter_section"][graves_plot_extracted_df["letter_section"]!=''].iloc[0:50])
# print(graves_plot_extracted_df["section"][graves_plot_extracted_df["section"]!=''].iloc[0:50])
# print(graves_plot_extracted_df["map"][graves_plot_extracted_df["map"]!=''].iloc[0:50])
# graves_plot_extracted_df["Merged"].iloc[0:50]

graves_plot_extracted_df["letter_section"].iloc[0:5]


In [ ]:
from IPython.display import display, HTML
graves_df = graves_df.dropna(subset=['plot'])
graves_df.reset_index(drop=True, inplace = True)
# graves_df[['cemeteryId','lastName','cemeteryName','plot']][graves_df['plot'].str.contains('Adas')]
graves_df_01=graves_df[['cemeteryId','lastName','cemeteryName','plot']]

display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))
pd.set_option('display.width', 200)
print(graves_df_01.iloc[0:50])


In [ ]:
def normalize_plot(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[*:,-]', ' ', text)        # replace separators with space
    text = re.sub(r'\s+', ' ', text)           # collapse multiple spaces
    return text.strip()


def extract_section(text):
    # match variations like "sec q", "sec-q", "section l"  
    pattern = r'(sec(?:tion)?)[\s\-\.:]*([a-z0-9]+)'
    match = re.search(pattern, text, re.I)
    # match = re.search(r'(sec(?:tion)?)[\s\-:]*([a-z0-9]+)', text, re.I)
    if match:
        return f"Section {match.group(2).upper()}"
    return None

graves_df_01=graves_df[['cemeteryId','lastName','cemeteryName','plot']]
graves_df_01 = graves_df_01.copy()
graves_df_01["normalized"] = graves_df_01["plot"].apply(normalize_plot)

graves_df_01["section"] = graves_df_01["normalized"].apply(extract_section)

graves_df_01_jewish_names = graves_df_01.merge(
    jewish_names_df,
    left_on=["lastName"],
    right_on=["lastName"],
    how="inner"
)

fileSectionsOut=DirMain+"Affilation_Analysis\\Outputs\\graves_df_01_jewish_names.csv"
graves_df_01_jewish_names.to_csv(fileSectionsOut,mode="w",index=False,header=True)


# phrases_df["jewish_section_norm"] = phrases_df["jewish_section"].str.lower()

# def match_jewish_section(row, phrases):
#     for keyword in phrases:
#         if keyword in row:
#             return keyword
#     return None

# keywords = phrases_df["jewish_section_norm"].unique().tolist()
# A["jewish_section_match"] = A["normalized"].apply(
#     lambda x: match_jewish_section(x, keywords)
# )
# print(A[["plot","normalized","section","jewish_section_match"]])

In [ ]:
def extract_section(text):
    # match variations like "sec q", "sec-q", "section l"
    
    pattern = r'(sec(?:tion)?)[\s\-\.:]*([a-z0-9]+)'
    match = re.search(pattern, text, re.I)
    # match = re.search(r'(sec(?:tion)?)[\s\-:]*([a-z0-9]+)', text, re.I)
    if match:
        return f"Section {match.group(2).upper()}"
    return None

graves_df["section"] = graves_df["plot_normalized"].apply(extract_section)


phrases_df["jewish_section_norm"] = phrases_df["jewish_section"].str.lower()

def match_jewish_section(row, phrases):
    for keyword in phrases:
        if keyword in row:
            return keyword
    return None

keywords = phrases_df["jewish_section_norm"].unique().tolist()
graves_df["jewish_section_match"] = graves_df["plot_normalized"].apply(
    lambda x: match_jewish_section(x, keywords)
)


merged = graves_df.merge(
    phrases_df,
    left_on=["cemeteryId", "jewish_section_match"],
    right_on=["cemetery_id", "jewish_section_norm"],
    how="left"
)

In [ ]:
print(graves_df_01['section'])